### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="rossmann_store_sales",
    dataset_year="2015",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/rossmann-store-sales/overview",
    download_description=r"""
kaggle competitions download -c rossmann-store-sales \
&& mkdir -p local-data-warehouse/rossmann_store_sales \
&& mv rossmann-store-sales.zip local-data-warehouse/rossmann_store_sales \
&& unzip local-data-warehouse/rossmann_store_sales/rossmann-store-sales.zip -d local-data-warehouse/rossmann_store_sales \
&& rm local-data-warehouse/rossmann_store_sales/rossmann-store-sales.zip
""",
    # References
    academic_reference_bibtex=r"""@misc{kaggle_rossmann_store_sales,
    title        = {Rossmann Store Sales},
    author       = {{Kaggle}},
    howpublished = {\url{https://www.kaggle.com/competitions/rossmann-store-sales/overview}},
    note         = {Kaggle competition page. Accessed: 2026-03-19},
    year         = {2015}
}
""",
    academic_reference_bibtex_key="kaggle_rossmann_store_sales",
    license="Kaggle License",
    data_tags=["Non-IID", "Temporal"],
    curation_comments="""
- We merge the store data with the train data.
- The data is from a Kaggle competition with a temporal split, so the test data cannot be used for unsupervised approachs.
- The winning solution is public: https://storage.googleapis.com/kaggle-forum-message-attachments/102102/3454/Rossmann_nr1_doc.pdf.
- The competition used 48 days for testing. The description says "Rossmann store managers are tasked with predicting their daily sales for up to six weeks in advance". Therefore, we define the same horizon for defining our test splits and add one day between train/test as a planning gap.
- We transform the date column to datetime format.
- We exclude days when the store is closed since they all have zero sales and are trivial to predict. We drop the Open column.
- We assign categorical data type to the Store column and the DayOfWeek column.
- For the train data, the # of customers per day is given, which is not available at the prediction point for the test data. The 1st-place solution says that one model variation computed recent-history features on the number of customers instead of sales, and also used store-level aggregates such as average sales per customer. However this would mean that we have to define split-specific splits. Since many high performing solutions did not utilize customer information, we drop the column.
- The StateHoliday column has two encodings for the entry 0, one as number and one as object. We combine them into one category.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Sales",
    problem_type="regression",
    objective_metric_name="rmse",
    time_on="Date",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

'''
NOTES
- We need to do some FE to make the "Customers" column useful. Test how much the full aggregated set differs from the splits.
'''
train = pd.read_csv(dataset_mold.path / "train.csv")
store = pd.read_csv(dataset_mold.path / "store.csv")
test = pd.read_csv(dataset_mold.path / "test.csv")

df = pd.merge(train, store, on="Store", how="left")
test = pd.merge(test, store, on="Store", how="left")

df["Date"] = pd.to_datetime(df["Date"])
test["Date"] = pd.to_datetime(test["Date"])

df = df[df["Open"] == 1].copy()
test = test[test["Open"] == 1].copy()

df = df.drop(columns=["Open"])
test = test.drop(columns=["Open"])

for col in ["Store", "DayOfWeek"]:
    df[col] = df[col].astype("category")
    test[col] = test[col].astype(df[col].dtype)

# Drop customer information. TODO: Enable split-specific data storage and feature engineering to utilize this information.
df = df.drop(columns=["Customers"])

df["StateHoliday"] = df["StateHoliday"].astype(str).astype("category")

as_cat_dtype = ["StoreType", "Assortment", "PromoInterval"]
df[as_cat_dtype] = df[as_cat_dtype].astype("category")

print("Loaded data shape:", df.shape)
df = df.sort_values("Date").reset_index(drop=True)

In [ ]:
df.Date.describe(),test.Date.describe()

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata

date_col = task_mold.time_on
target_col = task_mold.target_column_name

df = df.sort_values(by=date_col).reset_index(drop=True)

splits = {}
used_in_train = set()
used_in_test = set()
used_data = set()

last_date = df[date_col].max()

test_length = 42
planning_gap = 1

for split in range(n_splits * n_repeats):
    # walk backwards in 42-day blocks
    test_end_day = last_date - pd.DateOffset(days=split * test_length)
    test_start_day = test_end_day - pd.DateOffset(days=test_length - 1)

    # 1-day planning gap before test starts
    pred_point = test_start_day - pd.DateOffset(days=planning_gap)

    train_idx = df.loc[df[date_col] < pred_point].index
    test_idx = df.loc[
        (df[date_col] >= test_start_day) &
        (df[date_col] <= test_end_day)
    ].index

    assert len(set(train_idx).intersection(set(test_idx))) == 0, "Train and test should not overlap"

    if split > 0:
        prev_test_idx = splits[split - 1][0][1]
        assert len(set(test_idx).intersection(set(prev_test_idx))) == 0, "Test splits overlap"

    # if you expect daily data with no missing dates:
    test_dates = df.loc[test_idx, date_col]
    assert (test_dates.max() - test_dates.min()).days == test_length - 1, "Test window should span 42 days"

    print(f"\n=== Step {split} ===")
    print("Train size:", len(train_idx), "| Test size:", len(test_idx))
    print("Test start:", test_start_day, "| Test end:", test_end_day)
    print("Train target mean:", df.loc[train_idx, target_col].mean())
    print("Test target mean:", df.loc[test_idx, target_col].mean())

    splits[split] = {0: [train_idx, test_idx]}

    used_in_train.update(train_idx)
    used_in_test.update(test_idx)
    used_data.update(train_idx)
    used_data.update(test_idx)

print(f"{len(used_data)/df.shape[0]:.4f} of the samples are used.")
print(f"{len(used_in_train)/df.shape[0]:.4f} of the samples are used in training")
print(f"{len(used_in_test)/df.shape[0]:.4f} of the samples are used in testing.")

splits_mold = PredictiveMLSplitsMetadata( 
    splits_comment=f"The description says 'Rossmann store managers are tasked with predicting their daily sales for up to six weeks in advance'. \
        Therefore, we define the same horizon for our test splits and add one day between train/test as a planning gap.",
    splits=splits,
    time_horizon=test_length,
    time_horizon_unit="days",
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()